# LangChain中的消息（Messages）

本文基于 `langchain==1.3.0`，记录 LangChain 中 `Messages` 的基本用法。

主要内容：
- `Messages` 是什么，有什么用途
- 常见消息类型有哪些，分别如何使用
- 在 Agent 返回结果中，如何更友好地打印 `messages`
- 什么是多模态消息
- 如何使用百炼上的 `qwen3.5-plus` 分析在线图片和本地图片

参考：
- LangChain Messages 文档: https://docs.langchain.com/oss/python/langchain/messages
- LangChain Messages Reference: https://reference.langchain.com/python/langchain/messages
- LangChain Chat Models 文档: https://docs.langchain.com/oss/python/integrations/chat/
- 阿里云百炼视觉能力说明: https://help.aliyun.com/zh/model-studio/add-vision-skill


## 一、Messages 是什么

在 LangChain 中，聊天模型的输入和输出都围绕 `message` 展开。

可以把它理解成：**模型对话过程中的标准数据结构**。

它的主要作用有两点：
- 表达对话上下文，例如 system、user、assistant 之间的消息历史
- 统一不同 Provider 的聊天输入输出格式，便于 LangChain 在上层做抽象

官方文档的原始表述是：chat models take a sequence of messages as input and return messages as output。


## 二、Messages 的常见类型

LangChain 官方文档中，最常见的消息类型包括：
- `SystemMessage`
- `HumanMessage`
- `AIMessage`
- `ToolMessage`

其中前 3 个最常见，`ToolMessage` 主要出现在工具调用场景。


In [ ]:
# 导入 LangChain 中几种最常见的消息类型
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# SystemMessage：用于设定模型角色与行为
system_msg = SystemMessage("你是一个简洁的 AI 助手。")

# HumanMessage：表示用户输入
human_msg = HumanMessage("请用一句话解释 LangChain。")

# AIMessage：表示模型返回内容
ai_msg = AIMessage("LangChain 是一个用于构建 LLM 应用和 Agent 的开发框架。")

# ToolMessage：表示工具执行结果，通常要和 tool_call_id 对应
tool_msg = ToolMessage(
    content="上海今天天气晴，气温 22 度。",
    tool_call_id="call_123",
)

# 将不同类型的消息放入同一个数组中
messages = [system_msg, human_msg, ai_msg, tool_msg]
messages


### 1. `SystemMessage`

用于设置模型行为、角色、风格和限制。通常放在消息序列最前面。

### 2. `HumanMessage`

表示用户输入。既可以是纯文本，也可以包含图片、音频、文件等多模态内容。

### 3. `AIMessage`

表示模型返回结果。除了文本内容，还可能包含：
- `tool_calls`
- `usage_metadata`
- `response_metadata`

### 4. `ToolMessage`

表示工具执行后的结果，通常会回传给模型继续推理。


## 三、Messages 如何使用

在 LangChain 中，`messages` 最常见的用途有两类：
- 作为聊天模型的输入
- 作为 Agent 的输入与输出结构

下面使用百炼上的 `qwen3.5-plus` 作为示例模型，并通过 `Agent` 演示 `messages` 的传入和返回。


In [ ]:
# 模型初始化
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 加载 .env 中的百炼配置
load_dotenv()

# 这里通过 OpenAI 兼容端点初始化百炼上的多模态模型：qwen3.5-plus
model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.environ["DASHSCOPE_BASE_URL"],
    api_key=os.environ["DASHSCOPE_API_KEY"],
    temperature=0,
)


In [ ]:
# Agent 初始化
from langchain.agents import create_agent

# 这里不绑定工具，只演示 Agent 如何接收和返回 messages
agent = create_agent(model=model)

In [ ]:
# 创建一组对话消息，模拟多轮上下文
messages = [
    SystemMessage(content="你是一个简洁、准确的 AI 助手。"),
    HumanMessage(content="我的名字叫 peace"),
    AIMessage(content="你好 peace"),
    HumanMessage(content="请用三句话介绍 LangChain。"),
]

# 调用 Agent，发送整个 messages 数组
response = agent.invoke({"messages": messages})

# 直接打印整个 response，可读性通常较差
print(response)


上面的 `response` 本质上是一个字典，其中最重要的字段之一就是 `response["messages"]`。

如果直接打印整个 `response`，通常会包含较多 metadata，阅读成本较高。

LangChain 官方消息对象内置了 `pretty_print()`，可以直接按更友好的形式输出：


In [ ]:
# 遍历 Agent 返回的消息数组
for message in response["messages"]:
    # 使用 LangChain 内置的 pretty_print() 友好打印每条消息
    message.pretty_print()


这种方式的特点是：
- 不需要自己手写格式化函数
- 适合快速查看一轮对话中的消息结构
- 对 `SystemMessage`、`HumanMessage`、`AIMessage`、`ToolMessage` 都适用


## 四、多模态消息是什么

多模态消息指的是：一条消息中不仅包含文本，还可以包含图片、音频、视频、文件等其他类型的数据。

消息的 `content` 可以是：
1. 字符串、图片、视频、语音...
2. Provider 原生格式的内容块列表
3. LangChain 标准内容块列表

对于图片输入，推荐的标准内容块形式通常是：
- 在线图片：`{"type": "image", "url": ...}`
- 本地图片：`{"type": "image", "base64": ..., "mime_type": ...}`


In [ ]:
# 一条多模态 HumanMessage：同时包含文本和在线图片
online_image_message = HumanMessage(content=[
    {"type": "text", "text": "请描述这张图片的主要内容。"},
    {"type": "image", "url": "https://example.com/1.png"},
])

## 五、使用百炼 `qwen3.5-plus` 分析在线图片

阿里云百炼文档说明：`qwen3.5-plus` 原生支持视觉理解，可以直接处理图片输入。

这里仍然使用 LangChain 官方 `Messages` 文档中的标准内容块格式，但调用方式改为 `Agent`。


In [ ]:
# 构造一条包含“文本 + 在线图片”的多模态消息
messages = [
    HumanMessage(content=[
        {"type": "text", "text": "请描述这张图片的主要内容，并指出画面中的关键信息。"},
        {"type": "image", "url": "https://pic.rmb.bdstatic.com/bjh/3ea10928c6f1/250115/813605ccc30fbc2f25ccdd6bed6149a8.jpeg?x-bce-process=image/watermark,bucket_baidu-rmb-video-cover-1,image_YmpoL25ld3MvNjUzZjZkMjRlMDJiNjdjZWU1NzEzODg0MDNhYTQ0YzQucG5n,type_RlpMYW5UaW5nSGVpU01HQg==,w_50,text_QOmXqueLl-WoseS5kA==,size_50,x_38,y_38,interval_2,color_FFFFFF,effect_softoutline,shc_000000,blr_2,align_1"},
    ])
]

# 阻塞式输出：等待完整结果返回
# response = agent.invoke({"messages": messages})
# print(response["messages"][-1].content)

# 流式输出：边生成边打印模型返回内容
for token, metadata in agent.stream(
    {"messages": messages},
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)


## 六、使用百炼 `qwen3.5-plus` 分析本地图片

本地图片通常需要先转为 Base64，再放入消息内容块中。

LangChain 官方文档中的标准形式是：
- `type="image"`
- `base64=...`
- `mime_type="image/png"` 或 `image/jpeg`

消息构造完成后，同样通过 `agent.invoke(...)` 发送。


In [ ]:
# 图片文件转Base64 函数
import base64
import mimetypes
from pathlib import Path

def encode_image_to_base64(image_path: str) -> tuple[str, str]:
    # 解析本地图片路径
    path = Path(image_path)

    # 根据文件名推断 MIME 类型
    mime_type, _ = mimetypes.guess_type(path.name)
    if mime_type is None:
        mime_type = "image/png"

    # 将本地图片读取并编码为 Base64 字符串
    image_base64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    return image_base64, mime_type


In [ ]:
# 将这里替换为你自己的本地图片路径
local_image_path = "./assets/dongpo.jpg"

# 先把本地图片转成 Base64 及其 MIME 类型
image_base64, mime_type = encode_image_to_base64(local_image_path)

# 构造一条包含“文本 + 本地图片”的多模态消息
messages = [
    HumanMessage(content=[
        {"type": "text", "text": "请描述这张本地图片的主要内容，并提取其中可见的关键信息。"},
        {
            "type": "image",
            "base64": image_base64,
            "mime_type": mime_type,
        },
    ])
]

# 通过 Agent 发送本地图片消息
response = agent.invoke({"messages": messages})
print(response["messages"][-1].content)


## 七、总结

可以把这篇压缩成 4 个重点：
- `messages` 是 LangChain 中表达对话上下文的核心结构，也是 Agent 常见的输入输出结构
- 常见类型主要是 `SystemMessage`、`HumanMessage`、`AIMessage`、`ToolMessage`
- 多模态消息允许在一条消息里同时传文本和图片
- 即使使用的是 `Agent`，图片分析的核心仍然是先正确构造 `messages`，再通过 `agent.invoke(...)` 发送

补充说明：
- `qwen3.5-plus` 是否支持图片输入，取决于百炼当前模型能力；当前官方文档说明其支持视觉理解
- 如果模型不支持视觉能力，即使消息格式正确，也无法完成图片分析
- 在线图片更适合快速验证，多用于已有公网 URL 的场景；本地图片的 Base64 方式更适合直接处理本机文件

如果只记一句话：**在 LangChain 中，模型能力最终通过 `messages` 表达，而应用层通常更贴近 `Agent` 来组织调用。**
